In [ ]:
"""
Description (EN):
This script reads fire event data from 'fires_suggestion.csv', locates the corresponding
Sentinel‑2 L2A preview files, and downloads 10×10 km square images containing ALL spectral
bands. It filters by cloud cover (max 10 %), searches up to 60 days for a cloud‑free image,
and – if the primary image has too many missing pixels – creates a multi‑image composite.
All output GeoTIFFs are projected into the correct UTM zone for Bulgaria
(EPSG:32634 for 18°–24°E, EPSG:32635 for 24°–30°E).

Опис (BG):
Скриптът чете данни за пожари от 'fires_suggestion.csv', намира съответните Sentinel‑2 L2A
прегледни файлове и изтегля квадратни изображения 10×10 км с ВСИЧКИ спектрални канали.
Филтрира по облачност (макс. 10 %), търси до 60 дни безоблачно изображение и – ако основното
има твърде много липсващи пиксели – създава композит от няколко изображения.
Всички изходни GeoTIFF файлове са проектирани в правилната UTM зона за България
(EPSG:32634 за 18°–24°E, EPSG:32635 за 24°–30°E).
"""

import pandas as pd
import pystac_client
import planetary_computer
from odc.stac import stac_load
import yaml
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle
import json
import os
from rasterio.transform import from_bounds, from_origin
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.crs import CRS
from pyproj import Transformer
import math
from datetime import datetime, timedelta
import pytz
from scipy.ndimage import zoom
import warnings
import traceback
warnings.filterwarnings('ignore')

# ======================== PATHS ============================
INPUT_CSV = r'D:\data\master_thesis\input\fires_suggestion.csv'
PREVIEW_DIR = r'D:\data\master_thesis\exports\sentinel2_fire_images\preview'

# ======================== CONFIGURATION ====================
SQUARE_SIZE_KM = 10
MAX_CLOUD_COVER = 10
INITIAL_DATE_RANGE_DAYS = 30
WIDER_DATE_RANGE_DAYS = 60
MIN_DAYS_FROM_FIRE = 5
MAX_NODATA_PERCENTAGE = 5
ENHANCED_NODATA_THRESHOLD = 200       # below this are suspicious (reflectance bands > 1000)
MIN_COMPLEMENTARY_COVERAGE = 20
MAX_COMPLEMENTARY_IMAGES = 3

def get_utm_epsg(lon):
    """Return UTM EPSG code for Bulgaria."""
    if 18.0 <= lon <= 24.0:
        return 32634   # UTM zone 34N
    elif 24.0 < lon <= 30.0:
        return 32635   # UTM zone 35N
    else:
        zone = int((lon + 180) / 6) + 1
        return 32600 + zone

# ======================== HELPER FUNCTIONS ==================

def add_scale_bar(ax, transform, scale_km=2):
    """Draw a scale bar on the image."""
    try:
        y_pixels, x_pixels = ax.images[0].get_array().shape[:2]
        pixel_size_x = transform.a
        pixel_size_y = -transform.e
        pixel_size = (abs(pixel_size_x) + abs(pixel_size_y)) / 2
        scale_length_m = scale_km * 1000
        scale_length_pixels = scale_length_m / pixel_size
        x_pos = x_pixels * 0.95 - scale_length_pixels
        y_pos = y_pixels * 0.95
        rect = Rectangle((x_pos, y_pos), scale_length_pixels, y_pixels * 0.01,
                         facecolor='white', edgecolor='black', linewidth=2,
                         transform=ax.transData)
        ax.add_patch(rect)
        ax.text(x_pos + scale_length_pixels / 2, y_pos - y_pixels * 0.02,
                f'{scale_km} km',
                ha='center', va='top', color='white', fontweight='bold', fontsize=10,
                bbox=dict(boxstyle="round,pad=0.3", facecolor='black', alpha=0.7),
                transform=ax.transData)
    except Exception as e:
        print(f"  ⚠ Could not add scale bar: {e}")

def display_rgb_from_all_bands(image_path, title):
    """Display RGB from a GeoTIFF."""
    try:
        with rasterio.open(image_path) as src:
            band_count = src.count
            band_descriptions = [src.descriptions[i] if src.descriptions else f"Band_{i+1}"
                                 for i in range(band_count)]
            metadata = src.tags()
            print(f"  📊 Image has {band_count} bands: {band_descriptions}")

            red_band = green_band = blue_band = None
            for i, desc in enumerate(band_descriptions):
                if desc == 'B04': red_band = i + 1
                elif desc == 'B03': green_band = i + 1
                elif desc == 'B02': blue_band = i + 1

            if red_band and green_band and blue_band:
                red = src.read(red_band)
                green = src.read(green_band)
                blue = src.read(blue_band)

                total_pixels = red.size
                nodata_pixels = np.sum((red == 0) & (green == 0) & (blue == 0))
                nodata_percentage = (nodata_pixels / total_pixels) * 100
                print(f"  📊 Nodata/black pixels: {nodata_percentage:.1f}%")

                def stretch_band(band, clip_percentiles=(2, 98)):
                    band_masked = band[band > 0]
                    if len(band_masked) > 0:
                        p_low, p_high = np.percentile(band_masked, clip_percentiles)
                        if p_high > p_low:
                            return np.clip((band - p_low) / (p_high - p_low), 0, 1)
                    return band.astype(np.float32) / band.max() if band.max() > 0 else band.astype(np.float32)

                rgb_stretched = np.dstack([
                    stretch_band(red), stretch_band(green), stretch_band(blue)
                ])
                rgb_stretched = np.power(rgb_stretched, 0.8)

                fig, ax = plt.subplots(figsize=(14, 12))
                ax.imshow(rgb_stretched)
                add_scale_bar(ax, src.transform, scale_km=2)

                full_title = f"{title}\n"
                full_title += (f"Fire ID: {metadata.get('fire_id','?')} | "
                               f"Fire: {metadata.get('fire_date','?')} | "
                               f"Image: {metadata.get('actual_image_date','?')} | "
                               f"Days: {metadata.get('days_from_fire','?')} | "
                               f"Cloud: {metadata.get('cloud_cover','?')}% | "
                               f"Black: {nodata_percentage:.1f}%")
                if metadata.get('is_multi_composite') == 'yes':
                    full_title += f" | Composite: Yes ({metadata.get('composite_images_count','?')} imgs)"
                plt.title(full_title, fontsize=14, fontweight='bold', pad=20)
                plt.axis('off')
                plt.subplots_adjust(left=0.01, right=0.99, top=0.90, bottom=0.01)
                plt.show()
                print(f"✅ Displayed RGB from: {os.path.basename(image_path)}")
            else:
                print(f"❌ Could not find RGB bands. Available: {band_descriptions}")
    except Exception as e:
        print(f"❌ Error displaying image: {e}")
        traceback.print_exc()

def create_square_bbox(lat: float, lon: float, size_km: float = SQUARE_SIZE_KM):
    """Return (min_lon, min_lat, max_lon, max_lat) for a square area."""
    R = 6371.0
    lat_offset = (size_km / 2) / R * (180 / math.pi)
    min_lat = lat - lat_offset
    max_lat = lat + lat_offset
    lon_offset = (size_km / 2) / (R * math.cos(math.radians(lat))) * (180 / math.pi)
    min_lon = lon - lon_offset
    max_lon = lon + lon_offset
    return (min_lon, min_lat, max_lon, max_lat)

def create_enhanced_nodata_mask(band_data_list, nodata_threshold=ENHANCED_NODATA_THRESHOLD):
    """Enhanced nodata mask – use only reflectance bands (caller's responsibility)."""
    if not band_data_list:
        return None
    bands_stack = np.array(band_data_list)
    zero_mask = np.all(bands_stack == 0, axis=0).astype(np.uint8)
    combined_mask = np.zeros_like(bands_stack[0], dtype=np.uint8)
    for band_data in band_data_list:
        suspicious = band_data < nodata_threshold
        combined_mask[suspicious] += 1
    suspicious_threshold = len(band_data_list) * 0.6
    nodata_mask = (combined_mask > suspicious_threshold).astype(np.uint8)
    enhanced_mask = np.logical_or(nodata_mask, zero_mask).astype(np.uint8)

    nodata_count = np.sum(enhanced_mask)
    total_pixels = enhanced_mask.size
    nodata_pct = (nodata_count / total_pixels) * 100
    zero_pct = (np.sum(zero_mask) / total_pixels) * 100
    suspicious_pct = (np.sum(nodata_mask) / total_pixels) * 100
    print(f"    Enhanced nodata mask: {nodata_pct:.1f}% nodata ({nodata_count}/{total_pixels})")
    print(f"    Zero-only: {zero_pct:.1f}%, Suspicious low-value: {suspicious_pct:.1f}%")
    return enhanced_mask

def get_fire_date_from_dataframe(fire_id, extracted_df):
    """Extract fire date and coordinates from dataframe using fire ID."""
    try:
        fire_id_str = str(fire_id).strip()
        fire_id_int = None
        try:
            fire_id_int = int(fire_id_str.lstrip('0'))
        except ValueError:
            pass
        matching_row = None
        if fire_id_int is not None:
            matching_row = extracted_df[extracted_df['Fire_id'] == fire_id_int]
        if matching_row is None or matching_row.empty:
            matching_row = extracted_df[extracted_df['Fire_id'].astype(str) == fire_id_str]
        if (matching_row is None or matching_row.empty) and fire_id_int is not None:
            padded = f"{fire_id_int:03d}"
            matching_row = extracted_df[extracted_df['Fire_id'].astype(str).str.zfill(3) == padded]

        if matching_row.empty:
            print(f"  ⚠ Fire ID '{fire_id}' not found in dataframe")
            return None, None, None

        row = matching_row.iloc[0]
        fire_date = row['Date']
        if isinstance(fire_date, str):
            for fmt in ("%Y-%m-%d", "%Y%m%d", "%d/%m/%Y", "%Y-%m-%d %H:%M:%S"):
                try:
                    fire_date = datetime.strptime(fire_date, fmt)
                    break
                except ValueError:
                    continue
            else:
                print(f"  ⚠ Unrecognized date format: {fire_date}")
                return None, None, None
        elif isinstance(fire_date, pd.Timestamp):
            fire_date = fire_date.to_pydatetime()
        if fire_date.tzinfo is None:
            fire_date = fire_date.replace(tzinfo=pytz.UTC)
        lat = float(row['Lat'])
        lon = float(row['Lon'])
        print(f"  ✅ Found fire {fire_id}: {fire_date.strftime('%Y-%m-%d')} ({lat:.5f}, {lon:.5f})")
        return fire_date, lat, lon
    except Exception as e:
        print(f"Error getting fire data: {e}")
        return None, None, None

def find_multiple_complementary_images(catalog, fire_coords, fire_date, image_type,
                                       original_date, enhanced_nodata_mask,
                                       max_images=MAX_COMPLEMENTARY_IMAGES,
                                       search_range_days=30):
    center_lat, center_lon = fire_coords
    if image_type == "before":
        end_date = fire_date - timedelta(days=MIN_DAYS_FROM_FIRE)
        start_date = end_date - timedelta(days=search_range_days)
    else:
        start_date = fire_date + timedelta(days=MIN_DAYS_FROM_FIRE)
        end_date = start_date + timedelta(days=search_range_days)
    datetime_range = f"{start_date.strftime('%Y-%m-%d')}/{end_date.strftime('%Y-%m-%d')}"
    print(f"  🔍 Searching complementary {image_type} images: {datetime_range}")
    search = catalog.search(
        collections=["sentinel-2-l2a"],
        datetime=datetime_range,
        intersects={"type": "Point", "coordinates": [center_lon, center_lat]},
        query={"eo:cloud_cover": {"lt": MAX_CLOUD_COVER}}
    )
    items = list(search.items())
    if not items:
        return []
    candidates = []
    for item in items[:20]:
        coverage_info = test_image_coverage(item, fire_coords, enhanced_nodata_mask)
        if coverage_info and coverage_info['coverage_percentage'] >= MIN_COMPLEMENTARY_COVERAGE:
            candidates.append({
                'item': item,
                'coverage_percentage': coverage_info['coverage_percentage'],
                'cloud_cover': item.properties.get('eo:cloud_cover', 100),
                'date': item.datetime.replace(tzinfo=pytz.UTC),
                'days_from_fire': (item.datetime.replace(tzinfo=pytz.UTC) - fire_date).days
            })
    candidates.sort(key=lambda x: (-x['coverage_percentage'], x['cloud_cover'], abs(x['days_from_fire'])))
    selected = [c['item'] for c in candidates[:max_images]]
    return selected

def test_image_coverage(stac_item, fire_coords, nodata_mask):
    try:
        available_bands = list(stac_item.assets.keys())
        sentinel_bands = [b for b in available_bands if b.startswith('B') and b not in ['visual','preview']]
        if not sentinel_bands:
            return None
        cfg = yaml.safe_load("""
        sentinel-2-l2a:
          assets:
            "*":
              data_type: uint16
              nodata: 0
              unit: '1'
            "visual":
              data_type: uint8
              nodata: 0
              unit: '1'
        "*":
          warnings: ignore
        """)
        center_lat, center_lon = fire_coords
        bbox = create_square_bbox(center_lat, center_lon)
        test_band = 'B04' if 'B04' in sentinel_bands else sentinel_bands[0]
        ds = stac_load([stac_item], bands=[test_band], stac_cfg=cfg, bbox=bbox, chunks={})
        if ds.sizes['x'] == 0 or ds.sizes['y'] == 0:
            return None
        band_data = np.squeeze(ds[test_band].values)
        if band_data.shape != nodata_mask.shape:
            zoom_factors = (nodata_mask.shape[0] / band_data.shape[0],
                            nodata_mask.shape[1] / band_data.shape[1])
            band_data = zoom(band_data, zoom_factors, order=1)
        valid_data_mask = band_data > ENHANCED_NODATA_THRESHOLD / 2
        valid_in_nodata = np.sum((nodata_mask == 1) & valid_data_mask)
        total_nodata = np.sum(nodata_mask == 1)
        if total_nodata == 0:
            return {'coverage_percentage': 100}
        return {
            'coverage_percentage': (valid_in_nodata / total_nodata) * 100,
            'valid_pixels': valid_in_nodata,
            'total_nodata': total_nodata
        }
    except:
        return None

def create_multi_image_composite(main_item, complementary_items, sentinel_bands, bbox, cfg,
                                 enhanced_nodata_mask, fire_coords):
    """Create composite using multiple complementary images."""
    try:
        print(f"  🧩 Creating composite with {len(complementary_items)} complementary images")
        all_images_data = []
        all_image_info = []

        ds_main = stac_load([main_item], bands=sentinel_bands, stac_cfg=cfg, bbox=bbox, chunks={})
        main_bands = {}
        for band in sentinel_bands:
            if band in ds_main:
                data = np.nan_to_num(np.squeeze(ds_main[band].values), nan=0)
                main_bands[band] = data
        if not main_bands:
            return None
        all_images_data.append({
            'source': 'main', 'bands': main_bands,
            'cloud_cover': main_item.properties.get('eo:cloud_cover', 100),
            'date': main_item.datetime.replace(tzinfo=pytz.UTC)
        })
        all_image_info.append({
            'id': main_item.id, 'source': 'main',
            'cloud_cover': main_item.properties.get('eo:cloud_cover', 100),
            'date': main_item.datetime.strftime('%Y-%m-%d')
        })

        for i, comp_item in enumerate(complementary_items):
            ds_comp = stac_load([comp_item], bands=sentinel_bands, stac_cfg=cfg, bbox=bbox, chunks={})
            comp_bands = {}
            for band in sentinel_bands:
                if band in ds_comp:
                    data = np.nan_to_num(np.squeeze(ds_comp[band].values), nan=0)
                    ref_shape = list(main_bands.values())[0].shape
                    if data.shape != ref_shape:
                        zoom_factors = (ref_shape[0] / data.shape[0], ref_shape[1] / data.shape[1])
                        data = zoom(data, zoom_factors, order=1)
                    comp_bands[band] = data
            if comp_bands:
                all_images_data.append({
                    'source': f'comp_{i}', 'bands': comp_bands,
                    'cloud_cover': comp_item.properties.get('eo:cloud_cover', 100),
                    'date': comp_item.datetime.replace(tzinfo=pytz.UTC)
                })
                all_image_info.append({
                    'id': comp_item.id, 'source': f'comp_{i}',
                    'cloud_cover': comp_item.properties.get('eo:cloud_cover', 100),
                    'date': comp_item.datetime.strftime('%Y-%m-%d')
                })

        composite_bands = {}
        band_usage_stats = {}
        for band in sentinel_bands:
            if band in main_bands:
                composite = main_bands[band].copy()
                band_stack = [img['bands'][band] for img in all_images_data if band in img['bands']]
                nodata_positions = np.where(enhanced_nodata_mask == 1)
                total_nodata = len(nodata_positions[0])
                filled_count = 0
                for idx in range(total_nodata):
                    y, x = nodata_positions[0][idx], nodata_positions[1][idx]
                    if band_stack[0][y, x] > ENHANCED_NODATA_THRESHOLD / 2:
                        composite[y, x] = band_stack[0][y, x]
                        filled_count += 1
                        continue
                    for comp_idx in range(1, len(band_stack)):
                        if band_stack[comp_idx][y, x] > ENHANCED_NODATA_THRESHOLD / 2:
                            composite[y, x] = band_stack[comp_idx][y, x]
                            filled_count += 1
                            break
                fill_percentage = (filled_count / total_nodata) * 100 if total_nodata else 100
                composite_bands[band] = composite
                band_usage_stats[band] = {'filled': fill_percentage, 'total_nodata': total_nodata}
        total_filled = sum(s['filled'] * s['total_nodata'] for s in band_usage_stats.values())
        total_nodata_all = sum(s['total_nodata'] for s in band_usage_stats.values())
        overall_fill = (total_filled / total_nodata_all * 100) if total_nodata_all else 100
        print(f"     Overall fill: {overall_fill:.1f}%")
        return {
            'composite_bands': composite_bands,
            'band_usage_stats': band_usage_stats,
            'image_info': all_image_info,
            'overall_fill_percentage': overall_fill
        }
    except Exception as e:
        print(f"Error creating composite: {e}")
        return None

def get_stac_item_from_preview(preview_data, fire_coords=None, fire_date=None, image_type="before", widen_search=False):
    try:
        catalog = pystac_client.Client.open(
            'https://planetarycomputer.microsoft.com/api/stac/v1',
            modifier=planetary_computer.sign_inplace
        )
        search = catalog.search(collections=["sentinel-2-l2a"], ids=[preview_data['item_id']])
        items = list(search.items())
        if not items:
            return None
        item = items[0]
        cloud_cover = item.properties.get('eo:cloud_cover', 100)
        if cloud_cover > MAX_CLOUD_COVER:
            if fire_coords and fire_date:
                date_range = WIDER_DATE_RANGE_DAYS if widen_search else INITIAL_DATE_RANGE_DAYS
                alt = find_alternative_item(catalog, fire_coords, fire_date,
                                           preview_data['acquisition_date'].split('T')[0],
                                           image_type, date_range, is_wider_search=widen_search)
                if alt:
                    return alt
                elif not widen_search:
                    print("  🔍 Trying wider search...")
                    return get_stac_item_from_preview(preview_data, fire_coords, fire_date, image_type, widen_search=True)
            return None
        return item
    except Exception as e:
        print(f"Error fetching STAC item: {e}")
        return None

def find_alternative_item(catalog, fire_coords, fire_date, target_date_str, image_type, date_range_days, is_wider_search=False):
    center_lat, center_lon = fire_coords
    target_date = datetime.strptime(target_date_str, "%Y-%m-%d").replace(tzinfo=pytz.UTC)
    if image_type == "before":
        end_date = fire_date - timedelta(days=MIN_DAYS_FROM_FIRE)
        start_date = end_date - timedelta(days=date_range_days)
    else:
        start_date = fire_date + timedelta(days=MIN_DAYS_FROM_FIRE)
        end_date = start_date + timedelta(days=date_range_days)
    datetime_range = f"{start_date.strftime('%Y-%m-%d')}/{end_date.strftime('%Y-%m-%d')}"
    search = catalog.search(
        collections=["sentinel-2-l2a"],
        datetime=datetime_range,
        intersects={"type": "Point", "coordinates": [center_lon, center_lat]},
        query={"eo:cloud_cover": {"lt": MAX_CLOUD_COVER}}
    )
    items = list(search.items())
    if items:
        items.sort(key=lambda x: (abs((x.datetime.replace(tzinfo=pytz.UTC) - fire_date).days),
                                  x.properties.get('eo:cloud_cover', 100)))
        return items[0]
    return None

def extract_date_from_preview(preview_data):
    return preview_data['acquisition_date'].split('T')[0]

def reproject_to_utm(bands_stack, src_transform, src_crs, target_epsg, bbox_geo):
    """Reproject entire band stack to the target UTM CRS."""
    dst_crs_obj = CRS.from_epsg(target_epsg)
    src_crs_obj = CRS.from_string(src_crs) if isinstance(src_crs, str) else src_crs
    resolution = abs(src_transform.a)  # assume square pixels

    # Geographic bbox → target UTM extent
    transformer = Transformer.from_crs("EPSG:4326", dst_crs_obj, always_xy=True)
    min_lon, min_lat, max_lon, max_lat = bbox_geo
    corners = [(min_lon, min_lat), (min_lon, max_lat), (max_lon, min_lat), (max_lon, max_lat)]
    projected = [transformer.transform(lon, lat) for lon, lat in corners]
    xs = [p[0] for p in projected]
    ys = [p[1] for p in projected]
    dst_minx, dst_maxx = min(xs), max(xs)
    dst_miny, dst_maxy = min(ys), max(ys)

    dst_width = int(round((dst_maxx - dst_minx) / resolution))
    dst_height = int(round((dst_maxy - dst_miny) / resolution))

    # North-up: y pixel size negative
    dst_transform = from_origin(dst_minx, dst_maxy, resolution, -resolution)

    n_bands = bands_stack.shape[0]
    destination = np.zeros((n_bands, dst_height, dst_width), dtype=bands_stack.dtype)
    for i in range(n_bands):
        reproject(
            source=bands_stack[i],
            destination=destination[i],
            src_transform=src_transform,
            src_crs=src_crs_obj,
            dst_transform=dst_transform,
            dst_crs=dst_crs_obj,
            resampling=Resampling.bilinear
        )
    print(f"    Reprojected → UTM {target_epsg}, shape {destination.shape}")
    print(f"    Bounds: ({dst_minx:.1f},{dst_miny:.1f},{dst_maxx:.1f},{dst_maxy:.1f})")
    return destination, dst_transform

def get_reflectance_bands_only(all_bands, band_names):
    """Return only surface reflectance bands (B01-B12, B8A) for robust nodata detection."""
    reflectance_bands = [b for b in band_names if (b.startswith('B') and len(b) == 3) or b == 'B8A']
    out_data = []
    out_names = []
    for bname, bdata in zip(band_names, all_bands):
        if bname in reflectance_bands:
            out_data.append(bdata)
            out_names.append(bname)
    return out_data, out_names

def get_transform_from_dataset(ds):
    """Get affine transform from xarray dataset, falling back to 10m default."""
    try:
        if hasattr(ds, 'rio') and ds.rio.transform():
            transform = ds.rio.transform()
            resolution = abs(transform.a)
            print(f"    Transform from rio: {transform}")
            return transform, resolution
        # Manual
        x_res = float((ds.x[1] - ds.x[0]).values)
        y_res = float((ds.y[1] - ds.y[0]).values)
        transform = from_bounds(
            float(ds.x[0].values), float(ds.y[-1].values),
            float(ds.x[-1].values), float(ds.y[0].values),
            len(ds.x), len(ds.y)
        )
        print(f"    Transform derived manually: {transform}")
        return transform, abs(x_res)
    except Exception as e:
        print(f"    ⚠ Transform fallback (10 m): {e}")
        return rasterio.Affine(10, 0, 0, 0, -10, 0), 10

def extract_and_save_all_bands_with_multi_composite(
    stac_item, output_path, image_type, original_date_str, fire_coords, fire_date,
    fire_id, catalog=None, target_utm_epsg=None
):
    """Extract 10km square with ALL bands, output forced to target UTM."""
    try:
        print(f"Extracting 10km square from {stac_item.id}...")
        center_lat, center_lon = fire_coords
        bbox = create_square_bbox(center_lat, center_lon)
        min_lon, min_lat, max_lon, max_lat = bbox

        if target_utm_epsg is None:
            target_utm_epsg = get_utm_epsg(center_lon)
        print(f"  Target UTM: EPSG:{target_utm_epsg}")

        cfg = yaml.safe_load("""
        sentinel-2-l2a:
          assets:
            "*":
              data_type: uint16
              nodata: 0
              unit: '1'
            "visual":
              data_type: uint8
              nodata: 0
              unit: '1'
            "AOT":
              data_type: uint16
              nodata: 0
              unit: '1'
            "WVP":
              data_type: uint16
              nodata: 0
              unit: '1'
            "SCL":
              data_type: uint8
              nodata: 0
              unit: '1'
        "*":
          warnings: ignore
        """)

        available_bands = list(stac_item.assets.keys())
        sentinel_bands = [b for b in available_bands
                          if b.startswith(('B', 'AOT', 'SCL', 'WVP'))
                          and b not in ['visual', 'preview']]
        sentinel_bands.sort()
        print(f"  Extracting bands: {sentinel_bands}")

        ds = stac_load([stac_item], bands=sentinel_bands, stac_cfg=cfg,
                       bbox=[min_lon, min_lat, max_lon, max_lat], chunks={})
        if ds.sizes['x'] == 0 or ds.sizes['y'] == 0:
            print("  ✗ No data returned for this bbox")
            return False

        # --- Improved CRS detection ---
        if hasattr(ds, 'rio') and ds.rio.crs is not None:
            source_crs_obj = ds.rio.crs
        elif 'crs' in ds.attrs and ds.attrs['crs'] is not None:
            crs_val = ds.attrs['crs']
            source_crs_obj = CRS.from_string(crs_val) if isinstance(crs_val, str) else crs_val
        else:
            # Heuristic: if x coordinates are > 1000, assume projected (UTM)
            x_val = float(ds.x[0].values)
            if x_val > 1000:
                print("    CRS not found – assuming source is already target UTM")
                source_crs_obj = CRS.from_epsg(target_utm_epsg)
            else:
                source_crs_obj = CRS.from_epsg(4326)

        source_crs_epsg = source_crs_obj.to_epsg()
        print(f"  Source CRS: EPSG:{source_crs_epsg}")

        all_bands_data = []
        band_names = []
        for band in sentinel_bands:
            if band in ds:
                data = np.nan_to_num(np.squeeze(ds[band].values), nan=0)
                if np.max(data) == 0:
                    print(f"  ⚠ Band {band} is entirely zero – skipping")
                    continue
                all_bands_data.append(data)
                band_names.append(band)
            else:
                print(f"  ⚠ Band {band} not found in dataset")

        if not all_bands_data:
            print("  ✗ No valid bands loaded")
            return False

        refl_bands, refl_names = get_reflectance_bands_only(all_bands_data, band_names)
        if not refl_bands:
            print("  ✗ No reflectance bands – cannot reliably detect nodata")
            return False
        enhanced_nodata_mask = create_enhanced_nodata_mask(refl_bands)
        nodata_percentage = (np.sum(enhanced_nodata_mask) / enhanced_nodata_mask.size) * 100
        print(f"  Initial nodata (reflectance): {nodata_percentage:.1f}%")

        is_multi_composite = False
        composite_result = None
        if nodata_percentage > MAX_NODATA_PERCENTAGE and catalog:
            complementary_items = find_multiple_complementary_images(
                catalog, fire_coords, fire_date, image_type,
                original_date_str, enhanced_nodata_mask
            )
            if complementary_items:
                composite_result = create_multi_image_composite(
                    stac_item, complementary_items, sentinel_bands,
                    [min_lon, min_lat, max_lon, max_lat], cfg,
                    enhanced_nodata_mask, fire_coords
                )
                if composite_result and composite_result['composite_bands']:
                    is_multi_composite = True

        if is_multi_composite:
            final_bands = []
            final_names = []
            for band in sentinel_bands:
                if band in composite_result['composite_bands']:
                    final_bands.append(composite_result['composite_bands'][band])
                    final_names.append(band)
            overall_fill = composite_result['overall_fill_percentage']
        else:
            final_bands = all_bands_data
            final_names = band_names
            overall_fill = 100.0

        shapes = [b.shape for b in final_bands]
        if len(set(shapes)) > 1:
            target_shape = max(set(shapes), key=shapes.count)
            for i, b in enumerate(final_bands):
                if b.shape != target_shape:
                    zoom_factors = (target_shape[0] / b.shape[0], target_shape[1] / b.shape[1])
                    final_bands[i] = zoom(b, zoom_factors, order=1)

        bands_stack = np.stack(final_bands, axis=0)
        print(f"  Stack shape before transform: {bands_stack.shape}")

        transform_src, resolution_m = get_transform_from_dataset(ds)

        # Reproject only if source CRS differs from target UTM
        if source_crs_epsg != target_utm_epsg:
            print(f"  🔄 Reprojecting EPSG:{source_crs_epsg} → EPSG:{target_utm_epsg}")
            bands_stack, transform_out = reproject_to_utm(
                bands_stack, transform_src, source_crs_obj, target_utm_epsg, bbox
            )
            if bands_stack is None or np.max(bands_stack) == 0:
                print("  ❌ Reprojection produced empty/zero data – aborting")
                return False
        else:
            transform_out = transform_src
            print(f"  ✅ Already in target UTM – no reprojection")

        if np.max(bands_stack) == 0:
            print("  ❌ Final stack is entirely zero – not saved")
            return False

        print(f"  Final stack min={np.min(bands_stack)}, max={np.max(bands_stack)}, "
              f"non‑zero={(bands_stack > 0).sum()}/{bands_stack.size}")

        profile = {
            'driver': 'GTiff',
            'height': bands_stack.shape[1],
            'width': bands_stack.shape[2],
            'count': bands_stack.shape[0],
            'dtype': bands_stack.dtype,
            'crs': CRS.from_epsg(target_utm_epsg).to_wkt(),
            'transform': transform_out,
            'compress': 'deflate',
            'nodata': 0,
        }
        # Corrected final nodata computation using reflectance bands only
        final_refl = []
        for i, bname in enumerate(final_names):
            if bname in refl_names:
                final_refl.append(bands_stack[i])
        if final_refl:
            final_mask = create_enhanced_nodata_mask(final_refl)
            final_nodata_pct = (np.sum(final_mask) / final_mask.size) * 100 if final_mask is not None else 0
        else:
            final_nodata_pct = (np.sum(bands_stack[0] == 0) / bands_stack[0].size) * 100

        with rasterio.open(output_path, 'w', **profile) as dst:
            dst.write(bands_stack)
            for i, name in enumerate(final_names, 1):
                dst.set_band_description(i, name)

            actual_date = stac_item.datetime.strftime('%Y-%m-%d')
            actual_dt = stac_item.datetime.replace(tzinfo=pytz.UTC) if stac_item.datetime.tzinfo else stac_item.datetime.replace(tzinfo=pytz.UTC)
            days_from_fire = (actual_dt - fire_date).days
            time_direction = "before_fire" if actual_dt < fire_date else "after_fire"

            tags = {
                'fire_id': fire_id,
                'fire_lat': center_lat, 'fire_lon': center_lon,
                'fire_date': fire_date.strftime('%Y-%m-%d'),
                'original_target_date': original_date_str,
                'actual_image_date': actual_date,
                'days_from_fire': abs(days_from_fire),
                'fire_time_direction': time_direction,
                'image_type': image_type,
                'square_size_km': SQUARE_SIZE_KM,
                'cloud_cover': stac_item.properties.get('eo:cloud_cover', 'unknown'),
                'initial_nodata_percentage': f"{nodata_percentage:.1f}",
                'final_nodata_percentage': f"{final_nodata_pct:.1f}",
                'is_multi_composite': "yes" if is_multi_composite else "no",
                'target_utm_epsg': str(target_utm_epsg)
            }
            if is_multi_composite and composite_result:
                tags['composite_type'] = 'multi_image'
                tags['composite_images_count'] = str(len(composite_result['image_info']))
                tags['composite_overall_fill'] = f"{overall_fill:.1f}"
                for i, info in enumerate(composite_result['image_info']):
                    prefix = 'main' if i == 0 else f'comp_{i}'
                    tags[f'{prefix}_image_id'] = info['id']
                    tags[f'{prefix}_date'] = info['date']
                    tags[f'{prefix}_cloud_cover'] = str(info['cloud_cover'])
            dst.update_tags(**tags)

        file_size_mb = os.path.getsize(output_path) / (1024**2)
        print(f"  ✅ Saved: {os.path.basename(output_path)} ({file_size_mb:.1f} MB)")
        print(f"     UTM EPSG:{target_utm_epsg}, shape {bands_stack.shape}, "
              f"final nodata (reflectance): {final_nodata_pct:.1f}%")
        return True
    except Exception as e:
        print(f"❌ Error: {e}")
        traceback.print_exc()
        return False

def check_and_process_image(preview_data, preview_filename, fire_id, image_type,
                            fire_coords, fire_date, target_utm_epsg, attempt_wider_search=False):
    original_date_str = extract_date_from_preview(preview_data)
    stac_item = get_stac_item_from_preview(preview_data, fire_coords, fire_date, image_type)
    if not stac_item and attempt_wider_search:
        print(f"  🔍 Trying wider search (±{WIDER_DATE_RANGE_DAYS} days)...")
        stac_item = get_stac_item_from_preview(preview_data, fire_coords, fire_date, image_type, widen_search=True)
    if not stac_item:
        return None
    if stac_item.properties.get('eo:cloud_cover', 100) > MAX_CLOUD_COVER:
        print("  ❌ Cloud cover too high")
        return None

    actual_date = stac_item.datetime.strftime('%Y-%m-%d')
    output_filename = f'square_10km_allbands_{image_type}_{fire_id}_{actual_date.replace("-", "")}.tif'
    output_path = os.path.join(PREVIEW_DIR, output_filename)

    if os.path.exists(output_path):
        try:
            with rasterio.open(output_path) as src:
                _ = src.count
            print(f"  📁 File already exists: {output_filename}")
            display_title = f"{image_type.capitalize()} Fire {fire_id} - 10km Square | EXISTING"
            display_rgb_from_all_bands(output_path, display_title)
            return output_path
        except:
            print("  ⚠ Corrupt file, re‑downloading...")
            os.remove(output_path)

    print(f"  📥 Downloading: {output_filename}")
    catalog = pystac_client.Client.open(
        'https://planetarycomputer.microsoft.com/api/stac/v1',
        modifier=planetary_computer.sign_inplace
    )
    success = extract_and_save_all_bands_with_multi_composite(
        stac_item, output_path, image_type, original_date_str,
        fire_coords, fire_date, fire_id, catalog, target_utm_epsg
    )
    if success:
        display_title = f"{image_type.capitalize()} Fire {fire_id} - 10km Square | NEW"
        display_rgb_from_all_bands(output_path, display_title)
        return output_path
    return None

def main(extracted_df):
    print("Extracting 10km Squares from Sentinel-2 Images with ALL BANDS (UTM forced)")
    print("=" * 70)
    print(f"MAX CLOUD: {MAX_CLOUD_COVER}%, MIN DAYS FROM FIRE: {MIN_DAYS_FROM_FIRE}")
    print(f"NODATA THRESHOLD: {MAX_NODATA_PERCENTAGE}%, MULTI-COMPOSITE: up to {MAX_COMPLEMENTARY_IMAGES} images")
    print(f"INPUT CSV: {INPUT_CSV}")
    print(f"OUTPUT DIR: {PREVIEW_DIR}")
    print("=" * 70)

    required = ['Fire_id', 'Date', 'Lat', 'Lon']
    missing = [c for c in required if c not in extracted_df.columns]
    if missing:
        print(f"❌ Missing columns: {missing}")
        return

    os.makedirs(PREVIEW_DIR, exist_ok=True)

    try:
        all_files = os.listdir(PREVIEW_DIR)
        preview_files = [f for f in all_files if f.startswith('preview_') and f.endswith('.json')]
        if not preview_files:
            print(f"❌ No preview files in {PREVIEW_DIR}")
            return
    except Exception as e:
        print(f"❌ Cannot list directory: {e}")
        return

    fire_files = {}
    for f in preview_files:
        parts = f.split('_')
        if len(parts) >= 3:
            fire_id_clean = ''.join(filter(str.isdigit, parts[2].replace('.json', '')))
            if fire_id_clean:
                fire_files.setdefault(fire_id_clean, {'before': [], 'after': []})
                if 'before' in f.lower():
                    fire_files[fire_id_clean]['before'].append(f)
                elif 'after' in f.lower():
                    fire_files[fire_id_clean]['after'].append(f)

    print(f"\nFound {len(fire_files)} unique fire locations")
    successful = []

    for fire_id_str, files in fire_files.items():
        fire_date, lat, lon = get_fire_date_from_dataframe(fire_id_str, extracted_df)
        if not fire_date:
            continue
        target_epsg = get_utm_epsg(lon)
        print(f"\n{'='*50}\nProcessing Fire {fire_id_str} (EPSG:{target_epsg})\n{'='*50}")

        before_path = None
        if files['before']:
            with open(os.path.join(PREVIEW_DIR, sorted(files['before'])[0])) as f:
                before_preview = json.load(f)
            before_path = check_and_process_image(before_preview, files['before'][0], fire_id_str,
                                                  "before", (lat, lon), fire_date, target_epsg,
                                                  attempt_wider_search=True)
            if before_path:
                successful.append(f"before_{fire_id_str}")

        after_path = None
        if files['after']:
            with open(os.path.join(PREVIEW_DIR, sorted(files['after'])[0])) as f:
                after_preview = json.load(f)
            after_path = check_and_process_image(after_preview, files['after'][0], fire_id_str,
                                                 "after", (lat, lon), fire_date, target_epsg,
                                                 attempt_wider_search=True)
            if after_path:
                successful.append(f"after_{fire_id_str}")

        if before_path and after_path:
            print(f"  ✅ Both before/after exist for fire {fire_id_str}")

    square_files = [f for f in os.listdir(PREVIEW_DIR) if f.startswith('square_10km_allbands_')]
    print(f"\n{'='*70}\nPROCESSING COMPLETED!")
    print(f"Total 10km square files: {len(square_files)}")
    print(f"Successful extractions: {len(successful)}")

if __name__ == "__main__":
    df = pd.read_csv(INPUT_CSV)
    print("Columns in CSV:", df.columns.tolist())

    lat_col = 'Lat'
    lon_col = 'Lon'
    date_col = 'Date'
    fire_id_col = 'Fire_id'

    extracted_df = df[[fire_id_col, lat_col, lon_col, date_col]].copy()
    extracted_df = extracted_df.dropna()
    extracted_df[date_col] = pd.to_datetime(extracted_df[date_col], errors='coerce')
    print(f"Loaded and cleaned {len(extracted_df)} rows from CSV")

    main(extracted_df)